In [ ]:
import pandas as pd

In [ ]:
import os
os.listdir("/kaggle/input")

In [ ]:
os.listdir("/kaggle/input/datasets")

In [ ]:
os.listdir("/kaggle/input/datasets/azmaininqiadadeeb")

In [ ]:
for root, dirs,files in os.walk("/kaggle/input/datasets/azmaininqiadadeeb"):
    for file in files :
        print(os.path.join(root,file))
    

In [ ]:
path = "/kaggle/input/datasets/azmaininqiadadeeb/nasa-turbofan-jet-engine-dataset-csv/train_FD001_descriptive.csv"
df = pd.read_csv(path)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["engine"].nunique()

In [ ]:
df.isnull().sum()

In [ ]:
df["cycle"].describe()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.hist(df["RUL"], bins= 30)
plt.xlabel("Remaining useful Life")
plt.ylabel("Frequency")
plt.title("RUL Distribution")
plt.show()

In [ ]:
sensor = "LPC outlet temperature (°R)"

In [ ]:
df[sensor]

In [ ]:
plt.hist(df[sensor], bins = 30)
plt.xlabel("sensor")
plt.ylabel("Frequency")
plt.title("Sensor Value Distribution")
plt.show()

In [ ]:
engine_1 = df[df["engine"]==1]

In [ ]:
plt.plot(
    engine_1["cycle"],
    engine_1[sensor]
)
plt.xlabel("Operating Cycle")
plt.ylabel(sensor)
plt.title("Sensor Trend for engine 1")
plt.show()

In [ ]:
plt.boxplot(df[sensor].dropna())

plt.ylabel(sensor)
plt.title("Sensor Outlier Detection")

In [ ]:
Q1 = df[sensor].quantile(0.25)
Q3 = df[sensor].quantile(0.75)

IQR = Q3 -Q1

lower_limit = Q1 -1.5 *IQR
upper_limit = Q3 + 1.5 *IQR
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Limit:",lower_limit)
print("Upper Limit:",upper_limit)

In [ ]:
correlation = df.corr(numeric_only = True)

In [ ]:
correlation["RUL"].sort_values()

# L 7

In [ ]:
df.columns.tolist()

In [ ]:
sensor_columns = [
    "Fan inlet temperature (°R)",
    "LPC outlet temperature (°R)",
    "HPC outlet temperature (°R)"
]
sensor_data = df[sensor_columns].copy()
sensor_data.head()

# temp_diff = HPC outlet temp. - LPC outlet temp.

In [ ]:
sensor_data["temperature_difference"] = (
    sensor_data["HPC outlet temperature (°R)"]
    - sensor_data["LPC outlet temperature (°R)"]
)
sensor_data.head()


In [ ]:
df["temperature_difference"] = (
    df["HPC outlet temperature (°R)"]
    -df["LPC outlet temperature (°R)"]
)
df[
[
    
    "LPC outlet temperature (°R)",
    "HPC outlet temperature (°R)",
    "temperature_difference"
]
].head()


In [ ]:
df["HPC_temperature_rolling_mean"] = (
    df.groupby("engine")["HPC outlet temperature (°R)"]
    .transform(
        lambda x : x.rolling(window=3).mean()
    )
)
df[
[
    "engine",
    "cycle",
    "HPC outlet temperature (°R)",
    "HPC_temperature_rolling_mean"
    
]
].head(10)

In [ ]:
df["HPC_temperature_rolling_mean"].isnull().sum()

In [ ]:
df = df.dropna(
    subset=["HPC_temperature_rolling_mean"]
)
df.head()

In [ ]:
df[
[
    "engine",
    "cycle",
    "temperature_difference",
    "HPC_temperature_rolling_mean"
]
].head(8)

In [ ]:
correlation = df.corr(
    numeric_only = True
)
correlation.head()

In [ ]:
rul_correlation = correlation["RUL"].sort_values(
    ascending= False
)
rul_correlation

In [ ]:
features = [
    "cycle",
    "Fan inlet temperature (°R)",
    "LPC outlet temperature (°R)",
    "HPC outlet temperature (°R)",
    "temperature_difference",
    "HPC_temperature_rolling_mean"
]
X = df[features].copy()
X.head()

In [ ]:
X.describe()

In [ ]:
from sklearn.preprocessing import StandardScaler 


In [ ]:
scaler = StandardScaler()

In [ ]:
X_scaled = scaler.fit_transform(X)
X_scaled[:5]

In [ ]:
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=features,
    index=X.index
)
X_scaled_df.head()

In [ ]:
X_scaled_df.mean()

In [ ]:
X_scaled_df.std()

In [ ]:
comparison = pd.DataFrame({
    "Original_HPC_Temperature": X[
        "HPC outlet temperature (°R)"
    ].head(10).values,

    "Scaled_HPC_Temperature": X_scaled_df[
        "HPC outlet temperature (°R)"
    ].head(10).values
})

comparison

# L 8

In [ ]:
features = [
    "cycle",
    "Fan inlet temperature (°R)",
    "LPC outlet temperature (°R)",
    "HPC outlet temperature (°R)",
    "temperature_difference",
    "HPC_temperature_rolling_mean"
]
X = df[features].copy()
X.head(10)

In [ ]:
y = df["RUL"].copy()

In [ ]:
print('X Shape:', X.shape)
print("y Shape:",y.shape)

In [ ]:
X.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print ('X_train shape:', X_train.shape )
print ('X_test shape:', X_test.shape )
print ('y_train shape:', y_train.shape )
print ('y_test shape:', y_test.shape )

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=features,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=features,
    index=X_test.index
)

In [ ]:
X_train_scaled_df.head()

In [ ]:
X_test_scaled_df.head()

# L 9

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression()

In [ ]:
model.fit(X_train_scaled_df, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled_df)

In [ ]:
print("Predicted RUL values:")
print(y_pred[:10])

In [ ]:
comparision = pd.DataFrame({
    "Actual RUL" : y_test.values,
    "Preducted RUL": y_pred
})

comparision.head(10)

In [ ]:
print("Actual RUL shape:", y_test.shape)
print("Predicted RUL shape:", y_pred.shape)

# L 10

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_scaled_df, y_train)

linear_pred = model.predict(X_test_scaled_df)

print("Predicted RUL values:")
print(linear_pred[:10])

In [ ]:
from sklearn.tree import DecisionTreeRegressor

decision_tree_model = DecisionTreeRegressor(
    random_state = 42
)

decision_tree_model.fit(X_train_scaled_df, y_train)

decision_tree_pred = decision_tree_model.predict(X_test_scaled_df)

print("Decision Tree Predictions:")
print(decision_tree_pred[:10])
                                                 

In [ ]:
from sklearn.ensemble import RandomForestRegressor

random_forest_model = RandomForestRegressor(
    n_estimators = 100,
    random_state = 42
)

random_forest_model.fit(X_train_scaled_df, y_train)

random_forest_pred = random_forest_model.predict(
    X_test_scaled_df
)

print("Random Forest Predictions:")
print(random_forest_pred [:10])

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting_model = GradientBoostingRegressor(
    random_state = 42
)

gradient_boosting_model.fit(X_train_scaled_df, y_train)

gradient_boosting_pred = gradient_boosting_model.predict(X_test_scaled_df)

print(" Gradient Boosting Predictions:")
print(gradient_boosting_pred[:10])


In [ ]:
comparison = pd.DataFrame({
    "Actual RUL": y_test.values,
    "Linear Regression": linear_pred,
    "Decision Tree": decision_tree_pred,
    "Random Forest": random_forest_pred,
    "Gradient Boosting": gradient_boosting_pred
})

comparison.head(10)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 
import numpy as np


In [ ]:
mae_results = {
    "Linear Regression": mean_absolute_error(y_test, linear_pred),
    "Decision Tree": mean_absolute_error(y_test, decision_tree_pred),
    "Random Forest": mean_absolute_error(y_test, random_forest_pred),
    "Gradient Boosting": mean_absolute_error(y_test, gradient_boosting_pred)
}

for model_name, mae in mae_results.items():
    print(model_name, "MAE" , mae)

In [ ]:
rmse_results = {
    "Linear Regression": np.sqrt(mean_squared_error(y_test, linear_pred)),
    "Decision Tree": np.sqrt(mean_squared_error(y_test, decision_tree_pred)),
    "Random Forest": np.sqrt(mean_squared_error(y_test, random_forest_pred)),
    "Gradient Boosting": np.sqrt(mean_squared_error(y_test, gradient_boosting_pred))
}

for model_name, rmse in rmse_results.items():
    print(model_name, "RMSE" , rmse)

In [ ]:
r2_results ={
    "Linear Regression":r2_score(y_test, linear_pred),
    "Decision Tree":r2_score(y_test, decision_tree_pred),
    "Random Forest":r2_score(y_test, random_forest_pred),
    "Gradient Boosting": r2_score(y_test, gradient_boosting_pred)
}

for model_name, r2 in r2_results.items():
    print(model_name, "R²" , r2)

## Final Comparision

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE":[
        mae_results["Linear Regression"],
        mae_results["Decision Tree"],
        mae_results["Random Forest"],
        mae_results[ "Gradient Boosting"]
    ],
    "RMSE": [
        rmse_results["Linear Regression"],
        rmse_results["Decision Tree"],
        rmse_results["Random Forest"],
        rmse_results[ "Gradient Boosting"]
    ],
    "R²":[
        r2_results["Linear Regression"],
        r2_results["Decision Tree"],
        r2_results["Random Forest"],
        r2_results[ "Gradient Boosting"]
    ]
    
})

model_comparison

In [ ]:
lowest_mae = model_comparison.loc[
 model_comparison["MAE"].idxmin(), "Model"
]

lowest_rmse = model_comparison.loc[
 model_comparison["RMSE"].idxmin(), "Model"
]

highest_r2 = model_comparison.loc[
 model_comparison["R²"].idxmax(), "Model"
]

print ("Lowest MAE:", lowest_mae)
print("Lowest RMSE:", lowest_rmse)
print("Highest R²:",highest_r2 )

# L 11

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting_model = GradientBoostingRegressor(
    random_state = 42
)

gradient_boosting_model.fit(X_train_scaled_df, y_train)

gradient_boosting_pred = gradient_boosting_model.predict(X_test_scaled_df)

print(" Gradient Boosting Predictions:")
print(gradient_boosting_pred[:10])


In [ ]:
y_pred = gradient_boosting_pred
print("Predicted RUL values:")
print(y_pred[:10])

In [ ]:
comparison = pd.DataFrame({
    "Actual RUL": y_test.values,
    "Predicted RUL": y_pred
})

comparison.head(10)

In [ ]:
def get_risk_level(predicted_rul):
    if predicted_rul > 100:
        return "Low Risk"
    elif predicted_rul >=50:
        return "Medium Risk"
    else:
        return "Hight Risk"

        

In [ ]:
comparison["Risk Level"] = comparison["Predicted RUL"].apply(
     get_risk_level
)
comparison.head(10)

# L 12

In [ ]:
def get_risk_level(predicted_rul):
    if predicted_rul > 100:
        return "Low Risk"
    elif predicted_rul >=50:
        return "Medium Risk"
    else:
        return "Hight Risk"


In [ ]:
def get_health_level(predicted_rul):
    if predicted_rul > 100:
        return "Healthy"

    elif predicted_rul >= 50:
        return "Warning"

    else:
        return "Critical"
 

In [ ]:
comparison["Health Level"] = comparison["Predicted RUL"].apply(
     get_health_level
)
comparison.head(10)

In [ ]:
def get_maintenance_alert(health_level):

    if health_level == "Healthy":
        return "Normal Monitoring"

    elif health_level == "Warning":
        return "Maintenance Attention"

    else:
        return "Maintenance Attention Required"

In [ ]:
comparison["Maintenance Alert"] = comparison["Health Level"].apply(
     get_maintenance_alert
)
comparison.head(20)

# L 13

In [ ]:
import joblib

In [ ]:
joblib.dump(gradient_boosting_model,"gradient_boosting_model.pkl")

In [ ]:
import os

print(os.path.exists("gradient_boosting_model.pkl"))


In [ ]:
loaded_model = joblib.load(
    "gradient_boosting_model.pkl"
)

In [ ]:
loaded_pred = loaded_model.predict(X_test_scaled_df)

In [ ]:
loaded_pred[:10]

In [ ]:
original_pred = gradient_boosting_model.predict(X_test_scaled_df)
original_pred[:10]

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

joblib.dump(scaler,"scaler.pkl")

print("Scaler Saved Successfully")

